In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotnine as gg
import numpy as np
import scanpy as sc
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV
from scvi.model import JaxSCVI
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
import scipy.stats as st

from essential.utils import load_kegg_pathways

## load data

In [ ]:
tf_annotations = pd.read_csv(
    "/workspace/data/250516_TF_perturbseq/TF_pathway_annotation_gemini3.csv"
)
tf_annotations["pathway"].value_counts()

In [ ]:
kegg_pathways = load_kegg_pathways()
kegg_pathways.head()

In [ ]:
fitness_data_spacer = pd.read_csv(
    "/workspace/data/calvo2020_dcas9fitness/Supp_data2_log2FC.csv"
).rename(columns={"Unnamed: 0": "spacer"})
display(fitness_data_spacer.head())

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.h5ad"
)
adata.obs = adata.obs.merge(fitness_data_spacer, how="left", on="spacer")
adata.obs["spacer_has_fitness_data"] = adata.obs["spacer"].isin(fitness_data_spacer["spacer"])
adata.obs["spacer_is_control"] = adata.obs["target"] == "nontargeting"
adata.obs["spacer_is_valid"] = adata.obs["spacer_has_fitness_data"] | adata.obs["spacer_is_control"]

display(adata.obs["spacer_has_fitness_data"].value_counts())
display(adata.obs["spacer_is_control"].value_counts())
display(adata.obs["spacer_is_valid"].value_counts())

In [ ]:
n_obs = adata.obs["spacer"].value_counts().to_frame("n_obs").reset_index()
fitness_data_spacer = fitness_data_spacer.merge(n_obs, how="left", on="spacer")

In [ ]:
bins = np.arange(0, 50, 1)
fitness_data_spacer["n_obs"].hist(bins=bins)
plt.show()

In [ ]:
avg_fitness_per_n_obs = (
    fitness_data_spacer.groupby("n_obs")
    .agg(
        T4_mean=("T4", "mean"),
        T4_median=("T4", "median"),
        T4_q25=("T4", lambda x: x.quantile(0.25)),
        T4_q75=("T4", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

In [ ]:
(
    gg.ggplot(avg_fitness_per_n_obs, gg.aes(x="n_obs", y="T4_median"))
    + gg.geom_point()
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        fitness_data_spacer.loc[lambda x: x["n_obs"] <= 20], gg.aes(x="factor(n_obs)", y="T4")
    )
    + gg.geom_violin(position="dodge")
    + gg.geom_point(
        avg_fitness_per_n_obs.loc[lambda x: x["n_obs"] <= 20],
        gg.aes(x="n_obs", y="T4_median"),
        color="red",
    )
    + gg.geom_point(
        avg_fitness_per_n_obs.loc[lambda x: x["n_obs"] <= 20],
        gg.aes(x="n_obs", y="T4_mean"),
        color="red",
        shape="+",
        size=5,
    )
    + gg.theme_minimal()
    + gg.labs(
        x="# of capsules",
        y="Fitness (T+12h)",
        title="Relationship between # of capsules and fitness",
    )
)

In [ ]:
adata_ = adata[adata.obs["spacer_is_valid"] == True].copy()

adata_.X = adata_.layers["reads"]
sc.pp.normalize_total(adata_)
adata_.obsm["counts_per_median"] = adata_.X.copy()
sc.pp.log1p(adata_)
adata_.layers["log1p_cpmedian"] = adata_.X.copy()

adata_.X = adata_.layers["reads"]
sc.pp.normalize_total(adata_, target_sum=1e4)
adata_.layers["counts_per_1e4"] = adata_.X.copy()


adata_.obs["library_size"] = adata_.layers["reads"].sum(1).A1

In [ ]:
# (
#     adata_.layers["antisense_counts"].toarray().sum(),
#     adata_.layers["antisense_reads"].toarray().sum(),
# )

In [ ]:
prop_ambiant = (2.0 * adata_.layers["antisense_reads"].sum(1) / adata_.layers["reads"].sum(1)).A1
adata_.obs["prop_ambiant"] = prop_ambiant

In [ ]:
pd.set_option("display.max_columns", None)

# General data exploration

In [ ]:
spacer_info = []
adata_ctrl = adata_[adata_.obs["spacer_is_control"] == True].copy()
for spacer in tqdm(adata_.obs["spacer"].unique()):
    adata_spacer = adata_[adata_.obs["spacer"] == spacer].copy()

    target = adata_.obs.query("spacer == @spacer")["target"].unique()
    assert len(target) == 1
    target = target[0]

    is_control = target == "nontargeting"
    if target in adata_spacer.raw.var_names:
        expr_spacer_10k = adata_spacer[:, target].layers["counts_per_1e4"].toarray()
        expr_ctrl_10k = adata_ctrl[:, target].layers["counts_per_1e4"].toarray()
        kd_strength_10k = np.log2(
            (expr_spacer_10k.mean(0) + 1.0) / (expr_ctrl_10k.mean(0) + 1.0)
        ).item()
        expr_spacer_10k = expr_spacer_10k.mean(0).item()
        expr_ctrl_10k = expr_ctrl_10k.mean(0).item()

        library_size = adata_spacer.layers["reads"].sum().item()
        prop_ambiant = (
            2.0
            * adata_spacer.layers["antisense_reads"].sum().item()
            / adata_spacer.layers["reads"].sum().item()
        )

    else:
        kd_strength_10k = np.nan
        expr_spacer_10k = np.nan
        expr_ctrl_10k = np.nan
        library_size = np.nan
        prop_ambiant = np.nan
        target = np.nan

    spacer_info.append(
        {
            "spacer": spacer,
            "kd_strength": kd_strength_10k,
            "is_control": is_control,
            "expr_spacer_10k": expr_spacer_10k,
            "expr_ctrl_10k": expr_ctrl_10k,
            "library_size": library_size,
            "prop_ambiant": prop_ambiant,
            "gene": target,
        }
    )
spacer_info = pd.DataFrame(spacer_info)

# clustering-based analysis

## A. prelims

In [ ]:
JaxSCVI.setup_anndata(adata_, batch_key="rt_bc", layer="reads")
model = JaxSCVI(adata_)
model.train(
    batch_size=1024,
    early_stopping=True,
    early_stopping_monitor="elbo_validation",
    early_stopping_min_delta=0.0,
    early_stopping_patience=45,
    check_val_every_n_epoch=1,
)
z = model.get_latent_representation()

In [ ]:
adata_.obsm["X_scVI"] = z
sc.pp.neighbors(adata_, use_rep="X_scVI")
sc.tl.umap(adata_, min_dist=0.5)

In [ ]:
sc.pl.umap(adata_, color=["rt_bc", "T4"], vmin=-1, vmax=1, cmap="bwr")

In [ ]:
adata_hvg = adata_.copy()
adata_hvg.X = adata_hvg.layers["reads"]
sc.pp.highly_variable_genes(adata_hvg, n_top_genes=1000, flavor="seurat_v3")
adata_hvg = adata_hvg[:, adata_hvg.var["highly_variable"]].copy()
adata_hvg

In [ ]:
JaxSCVI.setup_anndata(adata_hvg, batch_key="rt_bc")
model = JaxSCVI(adata_hvg)
model.train(
    batch_size=1024,
    max_epochs=400,
    early_stopping=True,
    early_stopping_monitor="elbo_validation",
    early_stopping_min_delta=0.0,
    early_stopping_patience=45,
    check_val_every_n_epoch=1,
)
z = model.get_latent_representation()

In [ ]:
model.history["elbo_validation"].plot()
plt.show()

In [ ]:
adata_hvg.obsm["X_scVI"] = z
sc.pp.neighbors(adata_hvg, use_rep="X_scVI")

In [ ]:
sc.tl.umap(adata_hvg, min_dist=0.1)
sc.pl.umap(adata_hvg, color=["rt_bc", "T4"], vmin=-1, vmax=1, cmap="bwr")

In [ ]:
sc.set_figure_params(dpi=500)

In [ ]:
# sc.tl.leiden(adata_hvg, resolution=0.1, key_added="leiden_0.1")
# sc.tl.leiden(adata_hvg, resolution=0.2, key_added="leiden_0.2")
# sc.tl.leiden(adata_hvg, resolution=0.5, key_added="leiden_0.5")
sc.pl.umap(adata_hvg, color=["leiden_0.1", "leiden_0.2", "leiden_0.5"], vmin=-1, vmax=1, cmap="bwr")

In [ ]:
mapper = {
    "0": "control-like",
    "1": "control-like",
    "2": "control-like",
    "3": "essential",
    "4": "control-like",
    "5": "control-like",
    "6": "essential",
    "7": "essential",
    "8": "essential",
    "9": "essential",
}

mapper = pd.Series(mapper)
adata_hvg.obs["leiden_cluster_type"] = adata_hvg.obs["leiden_0.5"].map(mapper)
sc.pl.umap(adata_hvg, color=["leiden_cluster_type", "T4"], vmin=-1, vmax=1, cmap="bwr")

In [ ]:
adata_.obs

In [ ]:
adata_.obs.loc[:, "leiden_cluster_type"] = adata_hvg.obs["leiden_cluster_type"].values
adata_essential = adata_[adata_hvg.obs["T4"] <= -3.0].copy()

## B. analysis - gene-level & spacer-level

In [ ]:
# adata_.write_h5ad(
#     "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.processed.h5ad"
# )
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.processed.h5ad"
)
adata_essential = adata[adata.obs["T4"] <= -3.0].copy()

In [ ]:
sc.pp.neighbors(adata_essential, use_rep="X_scVI")
sc.tl.umap(adata_essential, min_dist=0.5)

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata, min_dist=0.1)

In [ ]:
adata.obs["UMAP_1"] = adata.obsm["X_umap"][:, 0]
adata.obs["UMAP_2"] = adata.obsm["X_umap"][:, 1]

In [ ]:
adata_focus = adata[adata.obs["target"].isin(["lpxA", "lpxD", "lpxB", "lpxK"])].copy()
adata_focus.obs["target"] = adata_focus.obs["target"].astype(str)
(
    gg.ggplot(gg.aes(x="UMAP_1", y="UMAP_2"))
    + gg.geom_point(adata.obs, size=0.1)
    + gg.geom_point(data=adata_focus.obs, mapping=gg.aes(color="target"))
)

In [ ]:
from scvi.external import MRVI

In [ ]:
adata.obs["target"] = adata.obs["target"].astype(str)

In [ ]:
MRVI.setup_anndata(adata, sample_key="target")
model = MRVI(adata)
model.train(max_epochs=400)

In [ ]:
u = model.get_latent_representation(give_z=False)

### Study of detectability vs technical variables

In [ ]:
working_adata = adata.copy()
cluster_reps = (
    working_adata.obs.groupby("gene")["leiden_cluster_type"]
    .value_counts(normalize=True)
    .to_frame("proportion")
    .reset_index()
)

gene_metadata = []
for gene in cluster_reps["gene"].unique():
    obs_subset = working_adata.obs.query("gene == @gene")

    n_obs = obs_subset.shape[0]
    t1 = obs_subset["T1"].mean()
    t2 = obs_subset["T2"].mean()
    t3 = obs_subset["T3"].mean()
    t4 = obs_subset["T4"].mean()
    prop_ambiant = obs_subset["prop_ambiant"].mean()
    lib_size = working_adata[obs_subset.index].layers["reads"].sum(1).mean().item()
    gene_metadata.append(
        {
            "gene": gene,
            "n_obs": n_obs,
            "prop_ambiant": prop_ambiant,
            "lib_size": lib_size,
            "T1": t1,
            "T2": t2,
            "T3": t3,
            "T4": t4,
        }
    )
gene_metadata = pd.DataFrame(gene_metadata)

In [ ]:
cluster_reps_spacer = (
    working_adata.obs.groupby("spacer")["leiden_cluster_type"]
    .value_counts(normalize=True)
    .to_frame("proportion")
    .reset_index()
)

spacer_metadata = []
for spacer in tqdm(cluster_reps_spacer["spacer"].unique()):
    obs_subset = working_adata.obs.query("spacer == @spacer")
    gene = obs_subset["gene"].unique()
    assert len(gene) == 1
    gene = gene[0]

    n_obs = obs_subset.shape[0]
    t1 = obs_subset["T1"].mean()
    t2 = obs_subset["T2"].mean()
    t3 = obs_subset["T3"].mean()
    t4 = obs_subset["T4"].mean()
    prop_ambiant = obs_subset["prop_ambiant"].mean()
    prop_cluster = (
        cluster_reps_spacer.query("spacer == @spacer")
        .query("leiden_cluster_type == 'essential'")["proportion"]
        .mean()
    )
    lib_size = working_adata[obs_subset.index].layers["reads"].sum(1).mean().item()
    spacer_metadata.append(
        {
            "gene": gene,
            "spacer": spacer,
            "n_obs": n_obs,
            "prop_ambiant": prop_ambiant,
            "prop_cluster": prop_cluster,
            "lib_size": lib_size,
            "T1": t1,
            "T2": t2,
            "T3": t3,
            "T4": t4,
        }
    )
spacer_metadata = pd.DataFrame(spacer_metadata)

In [ ]:
all_spacer_info = (
    cluster_reps_spacer.merge(spacer_metadata, on="spacer", how="left")
    .query("leiden_cluster_type == 'essential'")  # Only focus on this proportion
    .assign(
        is_detectable=lambda x: x["proportion"] >= 0.7,
        is_poorly_detectable=lambda x: x["proportion"] <= 0.3,
        detectability_status=lambda x: np.where(
            x["is_detectable"],
            "detectable",
            np.where(x["is_poorly_detectable"], "undetectable", "ambiguous"),
        ),
    )
)
essential_spacer_info = all_spacer_info.loc[lambda x: x["T4"] <= -3.0]

In [ ]:
# all_spacer_info_enough_obs = all_spacer_info.query("n_obs > 5")
all_spacer_info_enough_obs = all_spacer_info
all_spacer_info_enough_obs.loc[:, "proportion_cut"] = pd.cut(
    all_spacer_info_enough_obs["proportion"], 10
)

In [ ]:
(gg.ggplot(all_spacer_info_enough_obs) + gg.geom_point(gg.aes(x="proportion", y="T4")))

In [ ]:
(gg.ggplot(all_spacer_info_enough_obs) + gg.geom_boxplot(gg.aes(x="proportion_cut", y="T4")))

In [ ]:
essential_spacer_info

In [ ]:
(
    gg.ggplot(essential_spacer_info)
    + gg.geom_violin(gg.aes(x="detectability_status", y="lib_size", fill="detectability_status"))
    + gg.theme_minimal()
    + gg.theme(
        legend_position="none",
    )
    + gg.labs(
        x="sgRNA detectability",
        y="avg. capsule library size",
        title="Avg. library size vs detectability for essential genes",
    )
    + gg.scale_y_log10()
)

In [ ]:
(
    gg.ggplot(essential_spacer_info)
    + gg.geom_violin(
        gg.aes(x="detectability_status", y="prop_ambiant", fill="detectability_status")
    )
    + gg.theme_minimal()
    + gg.theme(
        legend_position="none",
    )
    + gg.labs(
        x="sgRNA detectability",
        y="avg. prop. ambient RNA",
        title="Avg. prop. ambient RNA vs detectability for essential genes",
    )
)

In [ ]:
(
    gg.ggplot(
        essential_spacer_info,
        gg.aes(x="T4", color="detectability_status", fill="detectability_status"),
    )
    + gg.stat_ecdf(
        geom="line",
        size=1.0,
    )
    + gg.labs(
        x="Fitness (T+12h)",
        y="empirical CDF",
        fill="sgRNA detectability",
        title="Fitness vs detectability for essential genes",
    )
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        all_spacer_info,
        gg.aes(x="T4", color="detectability_status", fill="detectability_status"),
    )
    + gg.stat_ecdf(
        geom="line",
        size=1.0,
    )
    + gg.labs(
        x="Fitness (T+12h)",
        y="empirical CDF",
        fill="sgRNA detectability",
        title="Fitness vs detectability for all genes",
    )
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        essential_spacer_info,
        gg.aes(x="prop_ambiant", color="detectability_status", fill="detectability_status"),
    )
    + gg.stat_ecdf(
        geom="line",
        size=1.0,
    )
    + gg.xlim(0, 0.3)
    + gg.theme_minimal()
)

In [ ]:
st.ks_2samp(
    essential_spacer_info.query("detectability_status == 'detectable'")["prop_ambiant"],
    essential_spacer_info.query("detectability_status == 'undetectable'")["prop_ambiant"],
)

In [ ]:
(
    gg.ggplot(
        essential_spacer_info,
        gg.aes(x="lib_size", color="detectability_status", fill="detectability_status"),
    )
    + gg.stat_ecdf(
        geom="line",
        size=1.0,
    )
    + gg.scale_x_log10()
    + gg.theme_minimal()
)

In [ ]:
# detectable_genes = essential_genes_info.query("detectability_status == 'detectable'")
# poorly_detectable_genes = essential_genes_info.query("detectability_status == 'undetectable'")

# detectable_essential_genes = detectable_genes["gene"].unique()
# undetectable_essential_genes = poorly_detectable_genes["gene"].unique()

# print("identifiable essential genes:")
# print(len(detectable_essential_genes))
# print(detectable_essential_genes)
# print("undetectable essential genes:")
# print(len(undetectable_essential_genes))
# print(undetectable_essential_genes)

### Study of detectability vs OPS measurements

In [ ]:
import os


working_dir = "/workspace/data/Eaton_2025/Data/lDE20_Imaging"
df = pd.read_csv(
    os.path.join(working_dir, "2024-01-25_lDE20_Steady_State_df_Estimators_wStats.csv")
)

df_ = df.query("Estimator == 'Mean (Robust)'")
df_ops_all = (
    df_.pivot(
        index=["sgRNA", "Gene", "N Mismatch", "Category"], columns="Variable(s)", values="Value"
    )
    .reset_index()
    .set_index(["sgRNA", "Gene"])
)


df_ops = df_ops_all.query("`N Mismatch` == 0").reset_index()
df_ops_controls = df_ops_all.query("`Category` != 'target'")

#### Agreement between OPS and fitness screen

In [ ]:
merged_spacer_info = fitness_data_spacer.merge(
    df_ops_all, left_on="spacer", right_on="sgRNA", how="inner"
)
merged_spacer_info

In [ ]:
(gg.ggplot(merged_spacer_info, gg.aes(x="Delta time (s)", y="T4")) + gg.geom_point())

In [ ]:
merged_spacer_info

In [ ]:
from sklearn.linear_model import LassoCV

model = LassoCV(cv=5)
y = merged_spacer_info["T4"].values

predictive_features = [
    "Delta time (s)",
    "Instantaneous Growth Rate: Volume",
    "Length",
    "Septum Displacement Length Normalized",
    "Width",
    "mCherry mean_intensity",
]
X = (
    merged_spacer_info[predictive_features]
    .fillna(merged_spacer_info[predictive_features].median())
    .values
)
model.fit(X, y)

In [ ]:
model.coef_

In [ ]:
(
    gg.ggplot(
        merged_spacer_info, gg.aes(x="Delta time (s)", y="mCherry mean_intensity", color="T4")
    )
    + gg.geom_point()
    + gg.scale_color_continuous(limits=[-10, 0])
)

#### Predictability vs OPS measurements

In [ ]:
essential_spacer_info_ = essential_spacer_info.merge(
    df_ops, left_on="gene", right_on="Gene", how="inner"
)

In [ ]:
(
    gg.ggplot(
        essential_spacer_info_,
        gg.aes(
            x="mCherry mean_intensity",
            color="detectability_status",
            fill="detectability_status",
        ),
    )
    + gg.stat_ecdf(
        geom="line",
        size=1.0,
    )
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        essential_spacer_info_,
        gg.aes(
            x="Instantaneous Growth Rate: Volume",
            color="detectability_status",
            fill="detectability_status",
        ),
    )
    + gg.stat_ecdf(
        geom="line",
        size=1.0,
    )
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        essential_spacer_info_,
        gg.aes(x="Delta time (s)", color="detectability_status", fill="detectability_status"),
    )
    + gg.stat_ecdf(
        geom="line",
        size=1.0,
    )
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        essential_spacer_info_,
        gg.aes(x="Length", color="detectability_status", fill="detectability_status"),
    )
    + gg.stat_ecdf(
        geom="line",
        size=1.0,
    )
    + gg.theme_minimal()
)

#### Question: are undetectable genes (with larger division rates) still distinguishable from controls?

In [ ]:
(
    gg.ggplot(
        essential_genes_info_with_metadata, gg.aes(x="detectability_status", y="expr_ctrl_10k")
    )
    + gg.geom_boxplot()
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        essential_genes_info_with_metadata, gg.aes(x="detectability_status", y="library_size")
    )
    + gg.geom_boxplot()
    + gg.theme_minimal()
)

# predictive-based analysis 

**Objective**: we aim to understand if fitness can be reliably predicted from gene expression. At the same time, there are different technical confounders that may affect the prediction, namely:
- limited sequencing depth: right now, capsules are shallow-sequenced, so the expression is not very reliable, especially for low-expressed genes.
- ambient RNA: priming is a delicate step in the protocol; there may be significant residual ambient RNA in the library.
The objective is to assess whether these two factors may affect the prediction.
- partial KD: the KO may not be perfect (for some reason, it seems like the KD, which should be efficient, still lead to some residual expression).

**Approach**
We consider two learning setups, both involving a simple linear regression model, regressing fitness (obtained from a different study for the exact same experimental protocol) on gene expression (obtained from the current assay).
As a first approach, we will consider a random split of the data into training and test sets, in one of the two ways:
1. spacer-level analysis: regress fitness (associated with the spacer) on gene expression averaged across all capsules sharing the same spacer (idea: denoise expression to study KO strength effect)
2. single-capsule level analysis: regress fitness (associated with the spacer) on gene expression (all genes). study other potential failure modes of fitness prediction: limited sequencing depth, ambient RNA.

We will then define the fitness predictability of a spacer as the residual (in absolute norm) of the linear model for the spacer.


spacer-level analysis:
for each held-out spacer, we will plot the prediction error vs the following factors:
- KO strength: $\log \frac{Expr_{gene}}{Expr_{ctrl}}$: assess whether the KO is imperfect for some reason
 

single-capsule level analysis:
- Ambient RNA: proportion of missense reads in a given capsule
- Sequencing depth: $\log$ library size of a given capsule


In [ ]:
layer_name = "log1p_cpmedian"
target_name = "T4"

In [ ]:
all_spacers = adata_.obs["spacer"].unique()

In [ ]:
# data prep: bulk data
X_bulk = []
y_bulk = []
all_spacers = adata_.obs["spacer"].unique()
for unique_spacer in tqdm(all_spacers):
    X_bulk.append(
        adata_[adata_.obs.query("spacer == @unique_spacer").index].layers[layer_name].mean(0)
    )
    y_bulk.append(adata_.obs.query("spacer == @unique_spacer")[target_name].mean())
X_bulk = np.array(X_bulk).squeeze(1)
y_bulk = np.array(y_bulk)

In [ ]:
test_spacers = int(len(all_spacers) * 0.5)
heldout_spacers = np.random.choice(all_spacers, size=test_spacers, replace=False)

X_bulk_test = X_bulk[np.isin(all_spacers, heldout_spacers)]
y_bulk_test = y_bulk[np.isin(all_spacers, heldout_spacers)]
X_bulk_train = X_bulk[~np.isin(all_spacers, heldout_spacers)]
y_bulk_train = y_bulk[~np.isin(all_spacers, heldout_spacers)]

In [ ]:
# lr - spacer-level
model_bulk = LinearRegression()
model_bulk.fit(X_bulk_train, y_bulk_train)
y_bulk_pred = model_bulk.predict(X_bulk_test)
y_bulk_pred_train = model_bulk.predict(X_bulk_train)

In [ ]:
abs_residuals_bulk = np.abs(y_bulk_test - y_bulk_pred)

In [ ]:
spacer_info_test = spacer_info.set_index("spacer").loc[heldout_spacers].copy()
spacer_info_test.loc[:, "abs_residuals_bulk"] = abs_residuals_bulk
spacer_info_test.loc[:, "fitness_gt"] = y_bulk_test
spacer_info_test.loc[:, "fitness_pred"] = y_bulk_pred
spacer_info_test.loc[:, "is_essential"] = spacer_info_test["fitness_gt"] <= -3.0
spacer_info_test

In [ ]:
(
    gg.ggplot(spacer_info_test, gg.aes(x="expr_ctrl_10k", y="kd_strength"))
    + gg.geom_point(size=0.7)
    + gg.theme_minimal()
    + gg.scale_x_log10()
    + gg.stat_smooth(method="loess", se=False, color="red")
    + gg.geom_hline(yintercept=0, color="black", linetype="dashed")
    + gg.labs(x="Mean ctrl expression of target(CP10K)", y="KD strength (LFC)")
)

In [ ]:
(
    gg.ggplot(spacer_info_test, gg.aes(x="fitness_gt", y="fitness_pred"))
    + gg.geom_point()
    + gg.theme_minimal()
    + gg.stat_smooth(method="lm", se=False, color="red")
    + gg.labs(
        x="exper. fitness (GT)",
        y="pred. fitness",
    )
)

In [ ]:
(
    gg.ggplot(spacer_info_test, gg.aes(x="expr_ctrl_10k", y="abs_residuals_bulk"))
    + gg.geom_point(alpha=0.1)
    + gg.theme_minimal()
)

In [ ]:
spacer_info_test_large_expr = spacer_info_test.query("expr_ctrl_10k > 10.0").copy()

(
    gg.ggplot(
        spacer_info_test_large_expr,
        gg.aes(x="kd_strength", y="abs_residuals_bulk"),
    )
    + gg.geom_point()
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        spacer_info_test, gg.aes(x="kd_strength", y="abs_residuals_bulk", color="is_essential")
    )
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        spacer_info_test.query("is_essential"), gg.aes(x="kd_strength", y="abs_residuals_bulk")
    )
    + gg.geom_point()
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        spacer_info_test.query("is_essential"), gg.aes(x="library_size", y="abs_residuals_bulk")
    )
    + gg.geom_point(alpha=0.1)
    + gg.theme_minimal()
    + gg.scale_x_log10()
)

In [ ]:
(
    gg.ggplot(spacer_info_test, gg.aes(x="prop_ambiant", y="abs_residuals_bulk"))
    + gg.geom_point(alpha=0.1)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(spacer_info_test, gg.aes(x="fitness_gt", y="fitness_pred", color="expr_ctrl_10k"))
    + gg.geom_point(alpha=0.1)
    + gg.theme_minimal()
    + gg.scale_color_gradientn(colors=["blue", "red"], limits=[0, 10.0])
)

In [ ]:
X = adata_.layers[layer_name]
X = X.toarray()
y = adata_.obs[target_name]
spacer_names = adata_.obs["spacer"]

is_cell_heldout = spacer_names.isin(heldout_spacers)
X_train, X_test = X[~is_cell_heldout], X[is_cell_heldout]
y_train, y_test = y[~is_cell_heldout], y[is_cell_heldout]
spacer_names_train, spacer_names_test = (
    spacer_names[~is_cell_heldout],
    spacer_names[is_cell_heldout],
)

In [ ]:
# lr - single-capsule level
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)

In [ ]:
random_spacers = heldout_spacers[:5]
random_spacers

In [ ]:
adata_test = adata_[is_cell_heldout].copy()
adata_test.obs["y_pred"] = y_pred
adata_test.obs["y_gt"] = y_test
adata_test.obs["abs_residuals"] = np.abs(y_test - y_pred)
adata_test.obs["sq_residuals"] = (y_test - y_pred) ** 2

In [ ]:
pred_errors = (
    adata_test.obs.groupby("spacer")["sq_residuals"].mean().dropna().sort_values().to_frame()
)
pred_errors.head()

In [ ]:
spacer_info_ = spacer_info.merge(pred_errors, left_on="spacer", right_index=True)
spacer_info_.head()

In [ ]:
(
    gg.ggplot(spacer_info_, gg.aes(x="expr_ctrl_10k", y="sq_residuals"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
)

In [ ]:
pd.reset_option("display.max_rows")

In [ ]:
spacer_info

In [ ]:
plot_df = adata_test.obs.merge(spacer_info_, on="spacer", how="left")
plot_df

In [ ]:
plot_df_avg = (
    plot_df.groupby(["spacer", "gene"])[
        [
            "y_gt",
            "y_pred",
            "prop_ambiant",
            "library_size",
            "expr_ctrl_10k",
            "sq_residuals_x",
            "kd_strength",
        ]
    ]
    .mean()
    .reset_index()
)
plot_df_avg

In [ ]:
(
    gg.ggplot(plot_df, gg.aes(x="y_gt", y="y_pred"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + gg.stat_smooth(method="lm", se=False, color="red")
)

In [ ]:
(
    gg.ggplot(plot_df_avg, gg.aes(x="y_gt", y="expr_ctrl_10k"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(plot_df_avg, gg.aes(x="expr_ctrl_10k", y="sq_residuals_x", color="y_gt"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(plot_df_avg, gg.aes(x="y_gt", y="sq_residuals_x"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(plot_df_avg, gg.aes(x="kd_strength", y="sq_residuals_x", color="y_gt"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(plot_df_avg, gg.aes(x="library_size", y="sq_residuals_x", color="y_gt"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(plot_df_avg, gg.aes(x="prop_ambiant", y="sq_residuals_x", color="y_gt"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
)

In [ ]:
plot_df_avg.query("y_gt < -3.0").sort_values("sq_residuals_x").tail(50)["gene"].unique()

In [ ]:
adata_test.obs

In [ ]:
(
    gg.ggplot(adata_test.obs.query("y_gt < -3.0"), gg.aes(x="library_size", y="sq_residuals"))
    + gg.geom_point(size=0.5)
    + gg.stat_smooth(method="loess", se=False, color="red")
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(adata_test.obs.query("y_gt < -3.0"), gg.aes(x="prop_ambiant", y="sq_residuals"))
    + gg.geom_point(size=0.5)
    + gg.stat_smooth(method="loess", se=False, color="red")
    + gg.theme_minimal()
)

In [ ]:
pd.set_option("display.max_rows", None)

In [ ]:
gene_errors = adata_test.obs.groupby("gene")["abs_residuals"].mean().sort_values()
display(gene_errors.head(50).index.tolist())

In [ ]:
(
    gg.ggplot(adata_test.obs, gg.aes(x="library_size", y="abs_residuals"))
    + gg.geom_point(size=0.5)
    + gg.stat_smooth(method="loess", se=False, color="red")
    + gg.theme_minimal()
)

In [ ]:
top_spacers = adata_test.obs["spacer"].value_counts().iloc[:5].index

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(7, 20))
for i, spacer in enumerate(top_spacers):
    adata_subset = adata_test[adata_test.obs["spacer"] == spacer].copy()
    plt.sca(axes[i, 0])
    plt.scatter(adata_subset.obs["prop_ambiant"], adata_subset.obs["abs_residuals"])
    plt.sca(axes[i, 1])
    plt.scatter(adata_subset.obs["library_size"], adata_subset.obs["abs_residuals"])
    plt.tight_layout()
plt.show()

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# plt.sca(axes[0])
# plt.scatter(adata_test.obs["prop_ambiant"], adata_test.obs["abs_residuals"], s=0.5)
# plt.sca(axes[1])
# plt.scatter(adata_test.obs["library_size"], adata_test.obs["abs_residuals"], s=0.5)
# plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plt.sca(axes[0])
plt.scatter(adata_subset.obs["prop_ambiant"], adata_subset.obs["abs_residuals"])
plt.sca(axes[1])
plt.scatter(adata_subset.obs["library_size"], adata_subset.obs["abs_residuals"])
plt.show()

In [ ]:
prop_ambiant
library_size

In [ ]:
adata_test = adata_[test_indices].copy()
residuals = y_gt - y_pred
abs_residuals = np.abs(residuals)
adata_test.obs["residuals"] = residuals
adata_test.obs["abs_residuals"] = abs_residuals
adata_test.obs["y_pred"] = y_pred
adata_test.obs["y_gt"] = y_gt


adata_train = adata_[train_indices].copy()
adata_train.obs["y_pred"] = y_pred_train
adata_train.obs["y_gt"] = y_train

In [ ]:
(
    gg.ggplot(adata_train.obs.query("library_size < 10000"), gg.aes(x="y_gt", y="y_pred"))
    + gg.geom_point(alpha=0.1)
    + gg.theme_minimal()
    + gg.stat_smooth(method="lm", se=False, color="red")
)

In [ ]:
(
    gg.ggplot(adata_test.obs.query("library_size < 10000"), gg.aes(x="y_pred", y="y_gt"))
    + gg.geom_point(alpha=0.1)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(adata_test.obs.query("library_size < 10000"), gg.aes(x="library_size", y="residuals"))
    + gg.geom_point(alpha=0.1)
    + gg.theme_minimal()
    + gg.stat_smooth(method="loess", se=False, color="red")
)

In [ ]:
(
    gg.ggplot(
        adata_test.obs.query("library_size < 10000"), gg.aes(x="library_size", y="abs_residuals")
    )
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + gg.stat_smooth(method="loess", se=False, color="red")
)

In [ ]:
(
    gg.ggplot(
        adata_test.obs.query("library_size < 10000"),
        gg.aes(x="n_obs_in_perturbation", y="abs_residuals"),
    )
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + gg.stat_smooth(method="loess", se=False, color="red")
)

In [ ]:
st.pearsonr(adata_test.obs["residuals"], adata_test.obs["library_size"])

In [ ]:
X_bulk = []
fitness_bulk = []
all_spacers = adata_.obs["spacer"].unique()
for unique_spacer in tqdm(all_spacers):
    X_bulk.append(
        adata_[adata_.obs.query("spacer == @unique_spacer").index].layers["log1p_cpmedian"].mean(0)
    )
    fitness_bulk.append(adata_.obs.query("spacer == @unique_spacer")["T2"].mean())
X_bulk = np.array(X_bulk)
fitness_bulk = np.array(fitness_bulk)

In [ ]:
X_bulk = X_bulk.squeeze(1)

In [ ]:
X_bulk_train, X_bulk_test, y_bulk_train, y_bulk_test = train_test_split(
    X_bulk, fitness_bulk, test_size=0.2, random_state=42
)
model = LinearRegression()
model.fit(X_bulk_train, y_bulk_train)
y_pred_bulk = model.predict(X_bulk_test)
y_gt_bulk = y_bulk_test

In [ ]:
X_bulk = []
fitness_bulk = []
all_spacers = adata_.obs["spacer"].unique()
for unique_spacer in tqdm(all_spacers):
    X_bulk.append(
        adata_[adata_.obs.query("spacer == @unique_spacer").index].layers["log1p_cpmedian"].mean(0)
    )
    fitness_bulk.append(adata_.obs.query("spacer == @unique_spacer")["T2"].mean())
X_bulk = np.array(X_bulk)
fitness_bulk = np.array(fitness_bulk)

In [ ]:
plot_df = pd.DataFrame({"y_gt": y_gt_bulk, "y_pred": y_pred_bulk})

(
    gg.ggplot(plot_df, gg.aes(x="y_gt", y="y_pred"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + gg.stat_smooth(method="lm", se=False, color="red")
)